# Evaluacion Transversal — Predicción de abandono de clientes en ventas y marketing


**Integrantes:**

- Sebastian Lagos
- Oscar Oreste

**Contexto**

Una empresa del área de ventas y marketing dispone de información demográfica,
comercial y de experiencia de sus clientes. El objetivo del proyecto es analizar
estos datos y desarrollar un modelo de clasificación que permita identificar
clientes con riesgo de abandono.

La unidad de análisis corresponde a cada cliente y la variable objetivo es
`churn`, donde:

- `0`: el cliente no abandonó.
- `1`: el cliente abandonó.

El resultado puede apoyar decisiones de retención, marketing y experiencia del
cliente.

## Objetivos del proyecto

### Objetivo general

Desarrollar una solución reproducible de ciencia de datos para identificar
clientes con riesgo de abandono mediante técnicas de limpieza, integración de
datos, análisis exploratorio, clasificación y visualización interactiva.

### Objetivos específicos

1. Auditar la estructura y calidad del dataset.
2. Aplicar reglas de limpieza reproducibles sin modificar el archivo original.
3. Integrar indicadores externos mediante una API.
4. Analizar las características asociadas al abandono.
5. Comparar modelos de clasificación mediante accuracy.
6. Interpretar los resultados utilizando una matriz de confusión.
7. Construir un dashboard interactivo con Plotly 

## 1. Importación de librerías

Se importan las librerías principales que se utilizarán para cargar, revisar y
trabajar con el dataset.

Las librerías de visualización, API y Machine Learning se incorporarán en las
secciones correspondientes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Carga del dataset

El dataset se encuentra almacenado en la carpeta `data/raw`.

Se utilizan dos rutas posibles para permitir que el notebook pueda ejecutarse
desde la raíz del repositorio o desde la carpeta `notebook`.

In [4]:
rutas_posibles = [
    Path("data/raw/sales_marketing_customer_dataset.csv"),
    Path("../data/raw/sales_marketing_customer_dataset.csv")
]

ruta_dataset = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró sales_marketing_customer_dataset.csv en data/raw."
    )

df = pd.read_csv(ruta_dataset)

print("Dataset cargado correctamente.")
print("Ruta utilizada:", ruta_dataset)

Dataset cargado correctamente.
Ruta utilizada: ..\data\raw\sales_marketing_customer_dataset.csv


## 3. Vista inicial del dataset

Se muestran las primeras filas para conocer la estructura general y observar
ejemplos de los datos disponibles.

In [5]:
df.head()

,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,is_premium_user,total_visits,avg_session_time,pages_per_session,email_open_rate,email_click_rate,total_spent,avg_order_value,discount_used,coupon_code,support_tickets,refund_requested,delivery_delay_days,payment_method,satisfaction_score,nps_score,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn
0,10001,Male,52.00,India,Berlin,2022-05-10 00:00:00,2024-12-31 00:00:00,Email,Tablet,Annual,1,7,13.90,5.42,0.67,0.26,559.52,65.25,0,NEW20,0,0,3,UPI,3.00,10,27.56,915.31,14,0
1,10002,NaN,35.00,Germany,Mumbai,2024-06-16 00:00:00,2024-05-07 00:00:00,Organic,Desktop,Monthly,0,19,5.11,5.35,0.70,0.37,356.49,48.47,1,NEW20,5,0,3,BKash,3.00,7,15.15,"2,079.96",11,0
2,10003,Female,27.00,Germany,London,2023-08-23 00:00:00,2024-04-28 00:00:00,Email,Mobile,Annual,1,18,9.74,3.59,0.47,0.44,689.33,77.82,0,NaN,1,0,2,UPI,5.00,6,13.51,"1,379.15",9,0
3,10004,Female,36.00,India,Mumbai,2024-01-28 00:00:00,2023-05-20 00:00:00,Facebook Ads,Tablet,Annual,1,16,9.64,2.95,0.58,0.37,445.43,71.71,0,NaN,0,0,2,PayPal,4.00,6,25.65,774.65,7,0
4,10005,Male,29.00,USA,Hamburg,2023-07-21 00:00:00,2024-04-07 00:00:00,Referral,Mobile,Monthly,0,12,7.79,2.41,0.05,0.16,686.29,44.99,1,NaN,2,1,4,BKash,3.00,1,12.39,87.68,11,0


## 4. Inspección inicial del dataset

Se revisan las dimensiones, los nombres de las columnas y los tipos de datos.

Esta inspección permite comprender la estructura antes de aplicar cualquier
limpieza o transformación.

In [6]:
print("Dimensión del dataset (filas, columnas):")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

Dimensión del dataset (filas, columnas):
(15000, 30)

Columnas del dataset:
['customer_id', 'gender', 'age', 'country', 'city', 'signup_date', 'last_purchase_date', 'acquisition_channel', 'device_type', 'subscription_type', 'is_premium_user', 'total_visits', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'coupon_code', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'payment_method', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'churn']

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 15000 non-null  int64  
 1   gender                      14262 non-null  str    
 2   age                         13800 non-null

## 5. Resumen estadístico inicial

Se calculan estadísticas descriptivas para las variables numéricas.

Esto permite observar sus valores mínimos, máximos, promedios y posibles
diferencias de escala.

In [7]:
print("Resumen estadístico de las columnas numéricas:")
display(df.describe().T)

Resumen estadístico de las columnas numéricas:


,count,mean,std,min,25%,50%,75%,max
customer_id,"15,000.00","17,500.50","4,330.27","10,001.00","13,750.75","17,500.50","21,250.25","25,000.00"
age,"13,800.00",35.20,10.33,-4.00,28.00,35.00,42.00,95.00
is_premium_user,"15,000.00",0.30,0.46,0.00,0.00,0.00,1.00,1.00
total_visits,"15,000.00",15.00,3.89,3.00,12.00,15.00,18.00,31.00
avg_session_time,"15,000.00",8.02,2.99,0.01,5.97,7.99,10.06,19.12
pages_per_session,"15,000.00",4.00,1.48,0.01,2.99,4.00,5.01,10.84
email_open_rate,"15,000.00",0.50,0.29,0.00,0.24,0.50,0.75,1.00
email_click_rate,"15,000.00",0.25,0.14,0.00,0.13,0.25,0.38,0.50
total_spent,"13,950.00",524.36,467.05,0.27,300.43,498.84,702.40,"15,910.43"
avg_order_value,"15,000.00",60.08,24.75,0.07,43.03,60.11,76.89,154.55


## 6. Revisión del identificador de clientes

La columna `customer_id` identifica a cada cliente. Se comprueba su cantidad de
valores únicos y si existen identificadores repetidos.

In [8]:
print("Cantidad de registros:", len(df))
print("Clientes únicos:", df["customer_id"].nunique())
print(
    "Identificadores duplicados:",
    df["customer_id"].duplicated().sum()
)

Cantidad de registros: 15000
Clientes únicos: 15000
Identificadores duplicados: 0


## 7. Distribución de la variable objetivo

La variable `churn` indica si un cliente abandonó o no la empresa:

- `0`: no abandonó;
- `1`: abandonó.

Se revisa la cantidad y el porcentaje de clientes pertenecientes a cada clase.

In [9]:
distribucion_churn = df["churn"].value_counts().sort_index()

porcentaje_churn = (
    df["churn"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

resumen_churn = pd.DataFrame({
    "Cantidad": distribucion_churn,
    "Porcentaje": porcentaje_churn
})

display(resumen_churn)

,Cantidad,Porcentaje
churn,,
0,12702,84.68
1,2298,15.32


## 8. Baseline de accuracy

Como existe una clase mayoritaria, se calcula el resultado que obtendría una
regla que predijera siempre la clase más frecuente.

Este valor se utilizará posteriormente como referencia para evaluar los modelos
de clasificación.

In [10]:
clase_mayoritaria = df["churn"].mode()[0]

baseline_accuracy = (
    df["churn"] == clase_mayoritaria
).mean()

print("Clase mayoritaria:", clase_mayoritaria)
print(f"Baseline de accuracy: {baseline_accuracy:.2%}")

Clase mayoritaria: 0
Baseline de accuracy: 84.68%


### Interpretación inicial

El dataset contiene 15.000 registros y cada fila representa un cliente.

La clase `0`, correspondiente a clientes que no abandonaron la empresa, es
considerablemente más frecuente que la clase `1`.

Por este motivo, una accuracy cercana al 84,68 % no necesariamente representa
un buen modelo, ya que ese resultado puede alcanzarse prediciendo siempre la
clase mayoritaria.

Los modelos deberán superar o aportar información adicional respecto de este
baseline y sus resultados serán complementados con una matriz de confusión.

## 9. Revisión de calidad de los datos

Antes de limpiar el dataset se revisan los principales problemas de calidad:

- registros duplicados;
- valores nulos;
- categorías inconsistentes;
- edades inválidas;
- fechas incoherentes;
- relación entre país y ciudad;
- valores extremos.

En esta etapa solo se detectan y cuantifican los problemas. No se modifica el
DataFrame original.

## 10. Detección de duplicados

Se revisan primero los registros completamente duplicados.

También se comprueba si existen clientes con exactamente los mismos datos,
ignorando el identificador `customer_id`.

In [11]:
duplicados_exactos = df.duplicated().sum()

duplicados_sin_id = (
    df.drop(columns="customer_id")
    .duplicated()
    .sum()
)

print("Duplicados exactos:", duplicados_exactos)
print("Duplicados ignorando customer_id:", duplicados_sin_id)

Duplicados exactos: 0
Duplicados ignorando customer_id: 0


## 11. Detección de valores nulos

Se calcula la cantidad y el porcentaje de valores faltantes por columna.

El porcentaje permite evaluar si conviene imputar, conservar una categoría
explícita o excluir una variable del análisis.

In [12]:
resumen_nulos = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje": (df.isnull().mean() * 100).round(2)
})

resumen_nulos = (
    resumen_nulos[resumen_nulos["Cantidad"] > 0]
    .sort_values("Cantidad", ascending=False)
)

display(resumen_nulos)

,Cantidad,Porcentaje
coupon_code,6133,40.89
age,1200,8.00
total_spent,1050,7.00
gender,738,4.92
satisfaction_score,702,4.68


### Interpretación de los valores nulos

Los valores faltantes no deben tratarse todos de la misma manera.

- `coupon_code` presenta una proporción alta de ausencia y podría representar
  clientes que no utilizaron cupón.
- `gender` puede conservarse como una categoría explícita de información no
  registrada.
- `age`, `total_spent` y `satisfaction_score` son variables numéricas y su
  imputación se realizará posteriormente dentro del pipeline de Machine
  Learning.

No se utilizará el promedio automáticamente para todas las variables, porque
los valores extremos pueden afectar considerablemente la media.

## 12. Revisión de variables categóricas

Se inspeccionan las variables de texto con pocas categorías para detectar
diferencias de escritura, espacios adicionales o categorías vacías.

Las columnas de fechas se excluyen de esta revisión porque serán analizadas por
separado.

In [13]:
columnas_fecha = ["signup_date", "last_purchase_date"]

columnas_categoricas = [
    columna
    for columna in df.select_dtypes(include="object").columns
    if columna not in columnas_fecha
    and df[columna].nunique(dropna=True) <= 15
]

for columna in columnas_categoricas:
    print(f"\nVariable: {columna}")
    display(
        df[columna]
        .value_counts(dropna=False)
        .to_frame("Cantidad")
    )


Variable: gender


C:\Users\slago\AppData\Local\Temp\ipykernel_1920\3046558885.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for columna in df.select_dtypes(include="object").columns


,Cantidad
gender,
Male,6844
Female,6686
NaN,738
Other,732



Variable: country


,Cantidad
country,
Germany,3072
India,3014
Bangladesh,2984
USA,2975
UK,2955


...

,Cantidad
subscription_type,
Monthly,7666
Annual,7334



Variable: coupon_code


,Cantidad
coupon_code,
NaN,6133
REF10,2995
SALE15,2986
NEW20,2886



Variable: payment_method


,Cantidad
payment_method,
UPI,3105
PayPal,3005
SEPA,2986
BKash,2971
Card,2933


## 13. Revisión de la edad

Se revisan los valores mínimos y máximos de `age`.

Las edades menores o iguales a cero se consideran claramente inválidas. Los
clientes menores de 18 o mayores de 80 se informan como observaciones, pero no
se eliminan automáticamente porque podrían ser registros reales.

In [14]:
resumen_edad = pd.DataFrame({
    "Indicador": [
        "Edad mínima",
        "Edad máxima",
        "Edades negativas",
        "Edad igual a cero",
        "Menores de 18 años",
        "Mayores de 80 años"
    ],
    "Cantidad o valor": [
        df["age"].min(),
        df["age"].max(),
        (df["age"] < 0).sum(),
        (df["age"] == 0).sum(),
        (df["age"] < 18).sum(),
        (df["age"] > 80).sum()
    ]
})

display(resumen_edad)

,Indicador,Cantidad o valor
0,Edad mínima,-4.00
1,Edad máxima,95.00
2,Edades negativas,3.00
3,Edad igual a cero,1.00
4,Menores de 18 años,516.00
5,Mayores de 80 años,30.00


### Interpretación de la edad

Solo las edades menores o iguales a cero se consideran inválidas sin requerir
una regla de negocio adicional.

En el siguiente commit estos valores se convertirán en nulos. La imputación de
la edad se realizará posteriormente mediante la mediana calculada únicamente
con los datos de entrenamiento.

## 14. Revisión de fechas

Las columnas de fechas se convierten temporalmente para comprobar:

- formatos inválidos;
- fecha mínima y máxima;
- clientes cuya última compra aparece antes de la fecha de registro.

En esta etapa no se reemplazan ni se sobrescriben las columnas originales.

In [15]:
signup_date_temp = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

last_purchase_date_temp = pd.to_datetime(
    df["last_purchase_date"],
    errors="coerce"
)

signup_invalidas = (
    df["signup_date"].notna()
    & signup_date_temp.isna()
).sum()

last_purchase_invalidas = (
    df["last_purchase_date"].notna()
    & last_purchase_date_temp.isna()
).sum()

secuencia_fecha_invalida = (
    signup_date_temp.notna()
    & last_purchase_date_temp.notna()
    & (last_purchase_date_temp < signup_date_temp)
)

print("Fechas de registro inválidas:", signup_invalidas)
print("Fechas de última compra inválidas:", last_purchase_invalidas)

print("\nRango de signup_date:")
print(signup_date_temp.min(), "a", signup_date_temp.max())

print("\nRango de last_purchase_date:")
print(last_purchase_date_temp.min(), "a", last_purchase_date_temp.max())

print(
    "\nÚltima compra anterior al registro:",
    secuencia_fecha_invalida.sum()
)

print(
    "Porcentaje de inconsistencias:",
    f"{secuencia_fecha_invalida.mean() * 100:.2f}%"
)

Fechas de registro inválidas: 0
Fechas de última compra inválidas: 0

Rango de signup_date:
2022-01-01 00:00:00 a 2024-09-26 00:00:00

Rango de last_purchase_date:
2023-01-01 00:00:00 a 2025-03-10 00:00:00

Última compra anterior al registro: 3762
Porcentaje de inconsistencias: 25.08%


### Interpretación de las fechas

Las fechas presentan un formato válido, pero existe una cantidad importante de
registros donde la última compra aparece antes de la fecha de inscripción.

Estas fechas no se corregirán inventando valores ni utilizando una fecha
promedio o mediana.

En la limpieza se creará una bandera de inconsistencia y la antigüedad solo se
calculará cuando el orden temporal sea válido.

## 15. Revisión de consistencia entre país y ciudad

Se revisa si una misma ciudad aparece asociada a diferentes países dentro del
dataset.

Para realizar una comprobación interna, se utiliza como referencia el país más
frecuente asociado a cada ciudad.

Esta regla permite detectar asociaciones inusuales, pero no constituye una
validación geográfica externa definitiva.

In [19]:
ubicaciones = (
    df[["country", "city"]]
    .dropna()
    .copy()
)

pais_frecuente_por_ciudad = (
    ubicaciones
    .groupby("city")["country"]
    .agg(lambda valores: valores.mode().iloc[0])
)

ubicaciones["pais_referencia_interno"] = (
    ubicaciones["city"]
    .map(pais_frecuente_por_ciudad)
)

ubicaciones["inconsistencia_interna"] = (
    ubicaciones["country"]
    != ubicaciones["pais_referencia_interno"]
)

cantidad_inconsistencias = (
    ubicaciones["inconsistencia_interna"]
    .sum()
)

porcentaje_inconsistencias = (
    ubicaciones["inconsistencia_interna"]
    .mean()
    * 100
)

print(
    "Combinaciones inusuales según la relación interna:",
    cantidad_inconsistencias
)

print(
    "Porcentaje:",
    f"{porcentaje_inconsistencias:.2f}%"
)


display(
    ubicaciones[
        ubicaciones["inconsistencia_interna"]
    ].head(10)
)

Combinaciones inusuales según la relación interna: 11818
Porcentaje: 78.79%


,country,city,pais_referencia_interno,inconsistencia_interna
2,Germany,London,USA,True
3,India,Mumbai,Germany,True
4,USA,Hamburg,UK,True
6,UK,New York,India,True
7,Germany,Berlin,India,True
9,Germany,Hamburg,UK,True
10,Germany,Delhi,USA,True
11,Bangladesh,New York,India,True
12,Germany,Hamburg,UK,True
13,India,Mumbai,Germany,True


### Interpretación de ubicación

La asociación dominante entre ciudad y país muestra una cantidad elevada de
combinaciones internas inusuales.

No se corregirá la ciudad mediante suposiciones. En el modelo inicial se
priorizará `country` y se evaluará excluir `city` debido a su inconsistencia.

La validación externa mediante la API se realizará por país, no por ciudad.

## 16. Revisión de valores extremos en total_spent

Se utiliza el rango intercuartílico para identificar valores extremos en
`total_spent`.

El método IQR define como posibles outliers los valores menores a:

Q1 - 1.5 × IQR

o mayores a:

Q3 + 1.5 × IQR

En esta etapa los registros solo se identifican. No se eliminan ni se truncan.

In [20]:
total_spent_sin_nulos = df["total_spent"].dropna()

q1_total_spent = total_spent_sin_nulos.quantile(0.25)
q3_total_spent = total_spent_sin_nulos.quantile(0.75)

iqr_total_spent = q3_total_spent - q1_total_spent

limite_inferior_total_spent = (
    q1_total_spent - 1.5 * iqr_total_spent
)

limite_superior_total_spent = (
    q3_total_spent + 1.5 * iqr_total_spent
)

filtro_outliers_total_spent = (
    df["total_spent"].notna()
    & (
        (df["total_spent"] < limite_inferior_total_spent)
        | (df["total_spent"] > limite_superior_total_spent)
    )
)

cantidad_outliers = filtro_outliers_total_spent.sum()

print("Mediana:", round(total_spent_sin_nulos.median(), 2))
print("Percentil 99:", round(total_spent_sin_nulos.quantile(0.99), 2))
print("Valor máximo:", round(total_spent_sin_nulos.max(), 2))

print("\nLímite inferior IQR:", round(limite_inferior_total_spent, 2))
print("Límite superior IQR:", round(limite_superior_total_spent, 2))

print("\nCantidad de valores extremos:", cantidad_outliers)
print(
    "Porcentaje:",
    f"{cantidad_outliers / len(df) * 100:.2f}%"
)

Mediana: 498.84
Percentil 99: 1216.01
Valor máximo: 15910.43

Límite inferior IQR: -302.51
Límite superior IQR: 1305.34

Cantidad de valores extremos: 78
Porcentaje: 0.52%


### Interpretación de los valores extremos

Los valores extremos de gasto no se eliminarán automáticamente, porque podrían
corresponder a clientes reales de alto valor.

En fases posteriores se evaluarán tres alternativas:

1. conservar la variable original;
2. crear una transformación logarítmica;
3. aplicar truncamiento únicamente si resulta necesario.

Cualquier límite utilizado para el modelo deberá calcularse con los datos de
entrenamiento y no con el dataset completo.

## 17. Revisión de variables constantes

Una variable constante contiene el mismo valor en todos los registros y no
aporta información para diferenciar clientes.

Se revisa si existen columnas con uno o ningún valor único.

In [21]:
cardinalidad = df.nunique(dropna=False)

variables_constantes = cardinalidad[
    cardinalidad <= 1
]

if variables_constantes.empty:
    print("No se detectaron variables constantes.")
else:
    print("Variables constantes detectadas:")
    display(variables_constantes.to_frame("Valores únicos"))

No se detectaron variables constantes.


## Conclusión de la auditoría de calidad

La auditoría permitió identificar problemas reales que justifican una etapa de
limpieza y transformación:

- existen valores nulos en variables numéricas y categóricas;
- se detectan edades menores o iguales a cero;
- las fechas tienen formato válido, pero existen secuencias temporales
  incoherentes;
- la relación entre ciudad y país presenta inconsistencias;
- `total_spent` contiene valores extremos;
- no se detectaron duplicados relevantes.

Durante esta etapa no se modificaron los datos.

En la siguiente fase se aplicarán reglas simples y justificadas:

- copia del DataFrame;
- conversión de edades inválidas en nulos;
- tratamiento explícito de categorías faltantes;
- conversión definitiva de fechas;
- creación de banderas de inconsistencia;
- creación de variables temporales válidas;
- preparación de las variables numéricas para su imputación posterior dentro
  del pipeline.